# BOJ OIS メインモデル

**このノートブックはメインモデルの単一の真実源（single source of truth）です。**

- 特徴量・ハイパーパラメータの変更があった場合は必ずこのノートブックを再実行し、結果を `designs/experiment_log.md` と `designs/current_model.md` に反映すること
- 実験用ノートブックはこのノートブックを **直接変更しない**。実験は `notebooks/archives/` や別のノートブックで行うこと

最終更新：2026-03-21  
仕様書：`designs/current_model.md`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, calculate_metrics

EXCEL_PATH   = '../data/BOJ_data.xlsx'
MEETING_PATH = '../data/BOJ_meeting_history.csv'
START_DATE   = '2024-01-01'

print('Setup complete')

## 1. データパイプライン

`designs/current_model.md` 記載の標準設定で実行する。

In [ ]:
# --- 標準パイプライン（メインモデル仕様）---
df_raw    = load_and_clean_data(EXCEL_PATH, MEETING_PATH)
df_feat   = generate_features(df_raw)           # d=0.4, window=50（デフォルト）
df_pooled = pool_boj_data(df_feat)              # Absolute_Meeting_ID・Days_since_first_seen を含む

print(f'総行数（プーリング後）: {len(df_pooled):,}')
print(f'期間: {df_pooled["Date"].min().date()} 〜 {df_pooled["Date"].max().date()}')
print(f'\n特徴量候補列の確認:')
check_cols = ['Absolute_Meeting_ID', 'Days_since_first_seen', 'M1_spread', 'M1_frac_diff']
for col in check_cols:
    exists = col in df_pooled.columns
    print(f'  {col}: {"OK" if exists else "MISSING"}')

## 2. Walk-forward 検証

In [ ]:
# 3d モデル（return_model=True で最終フォールドのモデルを取得）
res_3d, model_3d, X_test_3d, y_test_3d, X_train_3d = walk_forward_validation(
    df_pooled, 'Target_3d_norm', START_DATE, return_model=True
)

# 5d モデル
res_5d, model_5d, X_test_5d, y_test_5d, X_train_5d = walk_forward_validation(
    df_pooled, 'Target_5d_norm', START_DATE, return_model=True
)

print(f'3d: {len(res_3d):,} OOS サンプル, {res_3d["Fold"].nunique()} フォールド')
print(f'5d: {len(res_5d):,} OOS サンプル, {res_5d["Fold"].nunique()} フォールド')

## 3. OOS IC サマリー

In [ ]:
from src.modeling import summarize_ic

metrics_3d = summarize_ic(res_3d)
metrics_5d = summarize_ic(res_5d)

summary = pd.DataFrame([
    {'Horizon': '3d', 'Global IC': metrics_3d['ic_all'], 'CS IC': metrics_3d['cs_ic'], 'TS IC': metrics_3d['ts_ic'], 'Train IC': metrics_3d['train_ic'], 'Gap': metrics_3d['gap']},
    {'Horizon': '5d', 'Global IC': metrics_5d['ic_all'], 'CS IC': metrics_5d['cs_ic'], 'TS IC': metrics_5d['ts_ic'], 'Train IC': metrics_5d['train_ic'], 'Gap': metrics_5d['gap']}
])

print('=== OOS IC 総合サマリー ===')
print(summary.to_string(index=False))

fic_3d = pd.Series(metrics_3d['ic_by_fold'])
fic_5d = pd.Series(metrics_5d['ic_by_fold'])

In [ ]:
# フォールド別IC推移
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, fic, label in zip(axes, [fic_3d, fic_5d], ['3d', '5d']):
    if not fic.empty:
        fic.plot(kind='bar', ax=ax, color=['steelblue' if v >= 0 else 'tomato' for v in fic])
        ax.axhline(fic.mean(), color='red', linestyle='--', linewidth=1, label=f'平均={fic.mean():.3f}')
        ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{label} OOS IC by Fold')
    ax.set_xlabel('Fold')
    ax.set_ylabel('IC')
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Meeting_Index 別 IC

In [ ]:
def ic_by_meeting(res):
    return res.groupby('Meeting_Index').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0] if len(g) >= 10 else np.nan
    ).rename('IC')

mi_3d = ic_by_meeting(res_3d)
mi_5d = ic_by_meeting(res_5d)

mi_df = pd.DataFrame({'3d IC': mi_3d, '5d IC': mi_5d})
print('=== Meeting_Index 別 OOS IC ===')
print(mi_df.to_string())

mi_df.plot(kind='bar', figsize=(8, 4), title='Meeting_Index 別 IC')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel('Meeting_Index (1=次回会合)')
plt.ylabel('IC')
plt.tight_layout()
plt.show()

## 5. Feature Importance（最終フォールド）

In [ ]:
def plot_importance(model, title, top_n=20):
    imp = pd.Series(
        model.feature_importance(importance_type='gain'),
        index=model.feature_name()
    ).sort_values(ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(8, top_n * 0.35 + 1))
    imp[::-1].plot(kind='barh', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Gain')
    plt.tight_layout()
    plt.show()
    return imp

imp_3d = plot_importance(model_3d, '3d モデル Gain Importance（上位20）')
imp_5d = plot_importance(model_5d, '5d モデル Gain Importance（上位20）')

In [ ]:
# 上位特徴量の比較表
imp_comp = pd.DataFrame({
    '3d Gain': imp_3d.head(15),
    '5d Gain': imp_5d.reindex(imp_3d.head(15).index)
})
print('=== Gain Importance 上位15（3dソート）===')
print(imp_comp.to_string())

## 6. MPM 直前シグナル分析（Days_to_MPM ≤ 5）

In [ ]:
# Days_to_MPM をマージ
dtm = df_pooled[['Date', 'Meeting_Index', 'Days_to_MPM']].drop_duplicates()

def pre_mpm_analysis(res, label):
    r = pd.merge(res, dtm, on=['Date', 'Meeting_Index'], how='left')

    # バケット別IC
    buckets = [
        ('会合直前（≤5日）',  r['Days_to_MPM'] <= 5),
        ('会合前（6〜15日）', (r['Days_to_MPM'] > 5) & (r['Days_to_MPM'] <= 15)),
        ('通常期（>15日）',   r['Days_to_MPM'] > 15),
    ]
    rows = []
    for name, mask in buckets:
        sub = r[mask]
        if len(sub) < 10:
            continue
        ic, _ = spearmanr(sub['Actual'], sub['Pred'])
        # 大動き方向的中（実績上位25%）
        thr = sub['Actual'].abs().quantile(0.75)
        large = sub[sub['Actual'].abs() > thr]
        dir_acc = (np.sign(large['Actual']) == np.sign(large['Pred'])).mean() if len(large) > 0 else np.nan
        rows.append({'期間': name, 'サンプル数': len(sub), 'IC': round(ic, 4),
                     '大動き方向的中率': round(dir_acc, 4) if not np.isnan(dir_acc) else '-'})
    df_res = pd.DataFrame(rows)
    print(f'\n=== {label} MPM 直前シグナル ===')
    print(df_res.to_string(index=False))

pre_mpm_analysis(res_3d, '3d')
pre_mpm_analysis(res_5d, '5d')

---

## 結果の記録

このノートブックを実行したら、以下に OOS IC を記入して保存すること。

| 実行日 | 3d IC (Global/CS) | 5d IC (Global/CS) | 変更内容 |
|--------|-------|-------|----------|
| 2026-03-21 | 0.2267 / -0.0051 | 0.2280 / 0.0743 | Exp-D/E採用（Absolute_ID, Days_since_first_seen） |

→ 記録後に `designs/current_model.md` と `designs/experiment_log.md` も更新すること。